In [ ]:
# Install dependencies (run once per fresh environment, before any imports;
# restart the kernel afterwards if packages were upgraded)
%pip install -q -r ../requirements.txt

In [ ]:
# --- Setup (Step 0) ---
import sys
from pathlib import Path

# Make src/ importable whether run locally (cwd=src/) or from repo root
for p in (Path.cwd(), Path.cwd() / "src"):
    if (p / "text2sql").is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from text2sql import config as C

C.set_seed()
# HF token (gated Gemma 3) is checked with C.get_hf_token() right before loading models
print("repo root:", C.REPO_ROOT)
print("models:", list(C.MODELS))
print("runtime:", C.runtime_info())

# Step 1 – Data: Spider dev, prompts, difficulty

The cells below build `outputs/spider_dev.json`, the per-example file that every later step reads and writes. It is a copy of Spider `dev.json` (all 1034 examples, original keys kept) with two keys added to each entry:

- `prompt`: the zero-shot prompt from `src/text2sql/prompt_template.txt`, filled with the example's `db_id`, `question`, and its database schema (CREATE TABLE + 3 sample rows). The design rationale is in `docs/prompt_design.md`.
- `difficulty`: easy / medium / hard / extra, computed with the official Spider evaluation script.

No GPU is needed for this step.

## Step 1.1 – Download Spider dev + databases

This step downloads the official Spider 1.0 zip from Google Drive (the link on yale-lily.github.io/spider) with `gdown` and unpacks it to `data/spider/`. We need the zip, not just the GitHub JSON files, because it also contains the SQLite databases (`database/<db_id>/<db_id>.sqlite`). The databases are used for the sample rows in the prompt and later for Execution Accuracy. The download is skipped if the files are already there. If Google Drive refuses the download (quota), download the zip by hand and unzip it so that `data/spider/` holds `dev.json`, `tables.json` and `database/`.

In [ ]:
import json
import zipfile

import gdown


def spider_ready() -> bool:
    return all((C.SPIDER_DIR / f).exists() for f in ("dev.json", "tables.json", "database"))


if not spider_ready():
    if C.SPIDER_DIR.exists():
        raise RuntimeError(f"{C.SPIDER_DIR} exists but is incomplete; remove it and rerun this cell.")
    C.DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = C.DATA_DIR / "spider_data.zip"
    if not zip_path.exists():
        gdown.download(id=C.SPIDER_GDRIVE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as zf:  # skip the __MACOSX/ junk entries in the zip
        zf.extractall(C.DATA_DIR, members=[m for m in zf.namelist() if m.startswith("spider_data/")])
    (C.DATA_DIR / "spider_data").rename(C.SPIDER_DIR)
    zip_path.unlink()

if not spider_ready():
    raise FileNotFoundError(
        "Spider download failed. Download the zip from https://yale-lily.github.io/spider by hand "
        f"and unzip it so that {C.SPIDER_DIR} contains dev.json, tables.json and database/."
    )

dev = json.loads((C.SPIDER_DIR / "dev.json").read_text())
tables = {t["db_id"]: t for t in json.loads((C.SPIDER_DIR / "tables.json").read_text())}


def db_path(db_id: str) -> Path:
    return C.SPIDER_DIR / "database" / db_id / f"{db_id}.sqlite"


dev_dbs = sorted({ex["db_id"] for ex in dev})
missing = [d for d in dev_dbs if d not in tables or not db_path(d).exists()]
assert len(dev) == 1034, len(dev)
assert not missing, f"missing schema or .sqlite for: {missing}"
print(f"{len(dev)} dev examples over {len(dev_dbs)} databases")

## Step 1.2 – Download the official evaluation script

This step clones `taoyds/test-suite-sql-eval` (the official evaluation script for Spider) and checks out a pinned commit, so difficulty labels and metrics stay reproducible. We import its SQL parser (`process_sql`) and `Evaluator` here so that Step 1.5 can call the script's own difficulty function. The parser tokenizes with nltk, which needs the `punkt_tab` data.

In [ ]:
import subprocess

import nltk

if not (C.EVAL_SCRIPT_DIR / "evaluation.py").exists():
    subprocess.run(["git", "clone", "--quiet", C.EVAL_SCRIPT_REPO, str(C.EVAL_SCRIPT_DIR)], check=True)
subprocess.run(["git", "-C", str(C.EVAL_SCRIPT_DIR), "checkout", "--quiet", C.EVAL_SCRIPT_COMMIT], check=True)
nltk.download("punkt_tab", quiet=True)

if str(C.EVAL_SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(C.EVAL_SCRIPT_DIR))
from evaluation import Evaluator
from process_sql import Schema, get_schema, get_sql

print("eval script at", C.EVAL_SCRIPT_COMMIT[:7], "->", C.EVAL_SCRIPT_DIR)

## Step 1.3 – Schema representation (CREATE TABLE + 3 sample rows)

This step builds the `{schema}` text for each database. For every table it writes:
- a `CREATE TABLE` statement built from `tables.json`, with original table/column names, column types (`others` → `TEXT`), `PRIMARY KEY` and `FOREIGN KEY ... REFERENCES ...` lines;
- a comment block with the first 3 rows (`SELECT * FROM <table> LIMIT 3`), read from the SQLite file. This is the "Create Table + Select 3" format of Rajkumar et al. (2022).

Names that are not plain identifiers (e.g. `Official_ratings_(millions)`) are double-quoted, which is valid SQLite. Cell values are tab-separated: `NULL` stands for missing values, line breaks become spaces, and values longer than `MAX_CELL_CHARS` are truncated. If a table's columns in `tables.json` don't match the database, the cell falls back to `SELECT *` and counts it (expected: 0). There are only 20 dev databases, so each one is rendered once and cached in `schemas`.

In [ ]:
import re
import sqlite3

TYPE_MAP = {"others": "text"}  # tables.json type -> type shown in the DDL
fallbacks: list[str] = []  # tables where the explicit-column SELECT failed


def ident(name: str) -> str:
    """Name as written in the DDL: quoted only if it is not a plain identifier."""
    return name if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name) else '"' + name.replace('"', '""') + '"'


def fmt_cell(value) -> str:
    s = "NULL" if value is None else re.sub(r"[\t\r\n]+", " ", str(value))
    return s if len(s) <= C.MAX_CELL_CHARS else s[: C.MAX_CELL_CHARS] + "..."


def sample_rows(conn, table: str, columns: list[str]) -> tuple[list[str], list[tuple]]:
    quoted = ", ".join('"' + c.replace('"', '""') + '"' for c in columns)
    try:
        cur = conn.execute(f'SELECT {quoted} FROM "{table}" LIMIT {C.N_SAMPLE_ROWS}')
        return columns, cur.fetchall()
    except sqlite3.OperationalError:
        fallbacks.append(table)
        cur = conn.execute(f'SELECT * FROM "{table}" LIMIT {C.N_SAMPLE_ROWS}')
        return [d[0] for d in cur.description], cur.fetchall()


def render_schema(t: dict, db_file: Path) -> str:
    names, cols, types = t["table_names_original"], t["column_names_original"], t["column_types"]
    pks = {i for pk in t["primary_keys"] for i in (pk if isinstance(pk, list) else [pk])}
    conn = sqlite3.connect(f"file:{db_file}?mode=ro", uri=True)
    conn.text_factory = lambda b: b.decode("utf-8", errors="replace")  # a few DBs have bad UTF-8
    blocks = []
    for ti, table in enumerate(names):
        idx = [i for i, (tj, _) in enumerate(cols) if tj == ti]
        lines = [
            f"  {ident(cols[i][1])} {TYPE_MAP.get(types[i], types[i]).upper()}" + (" PRIMARY KEY" if i in pks else "")
            for i in idx
        ]
        lines += [
            f"  FOREIGN KEY ({ident(cols[a][1])}) REFERENCES {ident(names[cols[b][0]])}({ident(cols[b][1])})"
            for a, b in t["foreign_keys"]
            if cols[a][0] == ti
        ]
        header, rows = sample_rows(conn, table, [cols[i][1] for i in idx])
        sample = [
            "/*",
            f"{C.N_SAMPLE_ROWS} example rows:",
            f"SELECT * FROM {ident(table)} LIMIT {C.N_SAMPLE_ROWS};",
            "\t".join(header),
            *("\t".join(fmt_cell(v) for v in row) for row in rows),
            "*/",
        ]
        blocks.append(f"CREATE TABLE {ident(table)} (\n" + ",\n".join(lines) + "\n);\n" + "\n".join(sample))
    conn.close()
    return "\n\n".join(blocks)


schemas = {db_id: render_schema(tables[db_id], db_path(db_id)) for db_id in dev_dbs}
for db_id, text in schemas.items():  # one CREATE TABLE per table in tables.json
    assert text.count("CREATE TABLE ") == len(tables[db_id]["table_names_original"]), db_id
print(f"rendered {len(schemas)} schemas; SELECT * fallbacks: {len(fallbacks)} {fallbacks}")
print(schemas["concert_singer"])

## Step 1.4 – Add the `prompt` key

This step fills the plain-text template (`src/text2sql/prompt_template.txt`) for every dev example and stores the result under a new `prompt` key, next to `question` and `db_id`. The template has exactly three placeholders: `{db_id}`, `{schema}` (from Step 1.3) and `{question}`. All models will get this same prompt as a single user message. We check that no placeholder is left unfilled and that every prompt contains its own question.

In [ ]:
import statistics

template = C.PROMPT_TEMPLATE.read_text()

for ex in dev:
    ex["prompt"] = template.format(db_id=ex["db_id"], schema=schemas[ex["db_id"]], question=ex["question"])

for ex in dev:
    assert not any(p in ex["prompt"] for p in ("{db_id}", "{schema}", "{question}")), ex["question"]
    assert ex["question"] in ex["prompt"]

lengths = [len(ex["prompt"]) for ex in dev]
print(f"prompt length (chars): min {min(lengths)}, median {statistics.median(lengths):.0f}, max {max(lengths)}")
print("=" * 80)
print(dev[0]["prompt"])

## Step 1.5 – Add the `difficulty` key (official hardness)

This step labels each example easy / medium / hard / extra using the official script's `Evaluator.eval_hardness`. We follow `evaluation.py` exactly: parse the gold SQL with the database schema (`get_sql(Schema(get_schema(db)), query)`), then classify the parsed query by counting its SQL components (joins, GROUP BY, nesting, ...). The resulting counts must match the published Spider dev split (248 / 446 / 174 / 166). That confirms the labels were reproduced correctly.

In [ ]:
from collections import Counter

evaluator = Evaluator()
eval_schemas = {db_id: Schema(get_schema(str(db_path(db_id)))) for db_id in dev_dbs}

for ex in dev:
    ex["difficulty"] = evaluator.eval_hardness(get_sql(eval_schemas[ex["db_id"]], ex["query"]))

counts = Counter(ex["difficulty"] for ex in dev)
print({level: counts[level] for level in ("easy", "medium", "hard", "extra")})
assert counts == {"easy": 248, "medium": 446, "hard": 174, "extra": 166}, counts

## Step 1.6 – Save the augmented dataset

This step writes the augmented dev set to `outputs/spider_dev.json`. Each entry keeps the original Spider keys in their original order, with `prompt` and `difficulty` added at the end. Later steps add model outputs under a `responses` key in this same file. The file is written deterministically, so rerunning Step 1 gives an identical file (same sha256).

In [ ]:
import hashlib

C.DEV_DATASET.parent.mkdir(parents=True, exist_ok=True)
C.DEV_DATASET.write_text(json.dumps(dev, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

digest = hashlib.sha256(C.DEV_DATASET.read_bytes()).hexdigest()
print(f"wrote {len(dev)} examples -> {C.DEV_DATASET} ({C.DEV_DATASET.stat().st_size / 1e6:.1f} MB)")
print("sha256:", digest)